# ServerWatch presentation notebook (solo demo)

This is the step-by-step script for the recorded assessment, in the order of the
[assignment brief](assignment-brief.md). Each step has **what to do**, **what the
commands do**, and **what to say**. It is the notebook version of the
[presentation guide](presentation-guide.md), written for one presenter.

**How to use it**

- The shell commands run **on EC2 or your own terminal**, not in this notebook. Copy them from here.
- The small Python cells are a **rehearsal timer**. Run *Start the timer* when you
  start, then the `checkpoint(...)` cell at the start of each step. It shows whether you
  are ahead of or behind the 60-minute plan. Use it while rehearsing; keep the
  notebook off screen during the real recording.
- Replace `ELASTIC_IP` with the instance address.

Requirement status: [coverage tracker](project-plan.md#assignment-coverage-tracker).

## Rules from the brief to keep in mind

- At most **60 minutes**, starting from a **brand-new instance**, including SSH keys,
  security group and connection. Rehearse on a different instance.
- Be **visible and heard**, with a readable terminal font.
- **Explain the commands** you run and **why** you chose each solution.
- Show basic terminal work **by hand** (step 5 opens the Nginx site in nano). Prepared
  code is allowed; a demo that is only `bash setup.sh` is not enough.
- Watch the recording, check the link, then **stop the instance**. An open HTTP port
  exposes it to the internet.

> **Solo note.** The brief is written for pairs (G2: both students connect from their own
> computers). Ask your teacher how to show G2 alone. Step 3 below shows two separate
> accounts, each with its own key, which is the closest solo equivalent.

## Before you press record

- [ ] The code is merged to `main` and pushed. `git log -1 --oneline` noted.
- [ ] Two SSH key pairs ready (the EC2 key for `ubuntu`, and your own key for the second account). Know your public IP.
- [ ] Discord webhook URL ready to paste (never show it on screen).
- [ ] `serverwatch.dev` DNS open in Namecheap (Advanced DNS), A records at TTL 5 min (VG2).
- [ ] Only a few real certificates per week: rehearse VG2 with `SERVERWATCH_CERTBOT_STAGING=1`.
- [ ] Chrome **secure DNS** set to Cloudflare, so the router's DNS cache can't show an old IP.
- [ ] Terminal font enlarged, unrelated tabs and notifications closed.
- [ ] This notebook open on a second screen.

Never show `.env`, tokens, passwords, webhook URLs or private keys on screen. Copy a token
in a second, unrecorded window.

## Time plan

| # | Part | Requirement | Minutes |
|---|---|---|---:|
| 1 | Introduction and architecture | — | 3 |
| 2 | Launch the EC2 instance | G1 | 7 |
| 3 | SSH with two separate keys | G2 | 4 |
| 4 | Clone, copy setup with SCP, run setup | G3, G4 | 12 |
| 5 | Verify, Nginx by hand, public Swagger | G7, G8 | 6 |
| 6 | systemd and the database connection | G7, G9 | 4 |
| 7 | Cron backup and restore | G5 | 6 |
| 8 | Discord alert | G6 | 4 |
| 9 | Docker Compose and persistence | VG1 | 7 |
| 10 | Domain and HTTPS | VG2 | 5 |
| 10b | Optional: deploy after CI | VG4 bonus | (4) |
| 11 | Wrap-up and shutdown | — | 2 |

The plan uses the full 60 minutes; do 10b only if earlier parts ran short. Setup in
step 4 takes several minutes: use that time for the talking points.

### Rehearsal timer

In [ ]:
import time

# (step, name, planned minutes, optional)
PLAN = [
    ("1", "Introduction and architecture", 3, False),
    ("2", "Launch the EC2 instance (G1)", 7, False),
    ("3", "SSH with two separate keys (G2)", 4, False),
    ("4", "Clone, SCP and setup (G3, G4)", 12, False),
    ("5", "Verify, Nginx by hand, Swagger (G7, G8)", 6, False),
    ("6", "systemd and the database (G7, G9)", 4, False),
    ("7", "Cron backup and restore (G5)", 6, False),
    ("8", "Discord alert (G6)", 4, False),
    ("9", "Docker Compose and persistence (VG1)", 7, False),
    ("10", "Domain and HTTPS (VG2)", 5, False),
    ("10b", "Optional: deploy after CI (VG4)", 4, True),
    ("11", "Wrap-up and shutdown", 2, False),
]
LIMIT = 60
started = None
reached = {}  # step -> minutes since start


def start():
    """Start (or restart) the rehearsal clock."""
    global started
    started = time.monotonic()
    reached.clear()
    print(f"Timer started. Budget: {LIMIT} minutes.")


def checkpoint(step):
    """Run at the start of a step: compare elapsed time with the plan."""
    if started is None:
        start()
    elapsed = (time.monotonic() - started) / 60
    reached[step] = elapsed
    planned = 0
    for number, name, minutes, optional in PLAN:
        if number == step:
            break
        if not optional:
            planned += minutes
    else:
        raise ValueError(f"Unknown step {step!r}")
    late = elapsed - planned
    status = "behind" if late > 0.5 else "on time"
    print(f"Step {step}: {name}")
    print(f"  elapsed {elapsed:.1f} min, planned start {planned} min -> {abs(late):.1f} min {'late' if late > 0 else 'early'} ({status})")
    if optional and elapsed + minutes + 2 > LIMIT:
        print("  Skip this optional step: not enough time left.")
    print(f"  {LIMIT - elapsed:.1f} min left of {LIMIT}.")


def summary():
    """Show how long each step took, to adjust the next rehearsal."""
    if started is None:
        print("Run start() first.")
        return
    total = (time.monotonic() - started) / 60
    order = [number for number, *_ in PLAN if number in reached]
    print(f"{'Step':<5} {'Planned':>7} {'Actual':>7}")
    for i, number in enumerate(order):
        end = reached[order[i + 1]] if i + 1 < len(order) else total
        planned = next(minutes for n, _, minutes, _ in PLAN if n == number)
        print(f"{number:<5} {planned:>7} {end - reached[number]:>7.1f}")
    print(f"Total {total:.1f} min of {LIMIT}" + ("  OVER THE LIMIT" if total > LIMIT else ""))

**Start the timer** when you start the rehearsal:

In [ ]:
start()

---
## 1. Introduction (3 min)

**Say:**
- "ServerWatch monitors one Ubuntu server. A collector measures CPU, memory and disk every
  five seconds and sends them to a FastAPI API, which stores them in PostgreSQL. A React
  dashboard shows them, and I can open incidents."
- "I deploy it twice on the same EC2 host: natively with systemd and Nginx for the G
  criteria, and with Docker Compose for VG."
- "The API has three separate tokens: reader, collector and admin. Each process gets only
  the credential it needs."

```
Browser ──► Nginx (:80/:443) ──► FastAPI (127.0.0.1:8000) ──► PostgreSQL (loopback)
                                      ▲
                     collector (systemd, every 5 s)
```

In [ ]:
checkpoint("1")

## 2. Launch the EC2 instance — G1 (7 min)

In the AWS console ([full steps](ec2-manual-setup.md#2-launch-a-fresh-ubuntu-ec2-instance-aws-console)):

1. **Launch instance**, name it, choose **Ubuntu 24.04 LTS** and a free-tier eligible type.
2. Create or select the key pair for the first `ubuntu` login.
3. Security group: **SSH 22** from your IP `/32` only.
4. Launch, wait for the status checks, then allocate and associate an **Elastic IP**.

**Say:**
- "Ubuntu LTS gets security updates for years, and my scripts target it."
- "I only open SSH, and only from my own address. PostgreSQL 5432 and the API's 8000 are
  never opened. HTTP 80 is added once Nginx is in place."
- "The Elastic IP keeps the address fixed when the instance restarts, which the domain
  needs later."

In [ ]:
checkpoint("2")

## 3. SSH with two separate keys — G2 (4 min)

Connect with the EC2 key:

```bash
chmod 400 ~/keys/initial-ec2-key.pem          # SSH refuses keys others can read
ssh -i ~/keys/initial-ec2-key.pem ubuntu@ELASTIC_IP
```

The second account (`junior`) is created by setup in step 4. After that, give it its own
key by hand ([details](ubuntu-accounts.md#give-each-student-ssh-access)):

```bash
sudo install -d -o junior -g junior -m 700 /home/junior/.ssh
sudo -u junior nano /home/junior/.ssh/authorized_keys   # paste the PUBLIC key by hand
sudo chmod 600 /home/junior/.ssh/authorized_keys
```

Then from a second terminal (ideally a second computer): `ssh -i ~/.ssh/junior_key junior@ELASTIC_IP` and `id`.

| Command | What it does |
|---|---|
| `chmod 400` | Makes the private key readable only by you; SSH refuses it otherwise |
| `install -d … -m 700` | Creates `.ssh` owned by that user, closed to others |
| `authorized_keys` | The list of public keys allowed to log in as this user |
| `id` | Shows the user and its groups, i.e. what it may do |

**Say:**
- "Every person gets their own key pair. Private keys never leave the computer; the server
  only gets the public key. Access can be removed per person by deleting one line."
- "`id` shows the junior account's groups: it can write to dev but only read prod."

In [ ]:
checkpoint("3")

## 4. Clone, SCP and setup — G3, G4 (12 min)

On EC2 as `ubuntu`:

```bash
sudo apt-get update && sudo apt-get install -y git
git clone https://github.com/EmirU116/server-api.git ~/serverwatch
cd ~/serverwatch && git log -1 --oneline
```

On your computer, from the checkout, send the setup script:

```bash
scp -i ~/keys/initial-ec2-key.pem setup.sh ubuntu@ELASTIC_IP:~/setup.sh
```

Save the Discord webhook privately ([commands](discord-alerts.md#configure-the-alert-discord-bash-systemd);
the prompt hides the input), then run setup on EC2:

```bash
export SERVERWATCH_SENIOR_USER=senior
export SERVERWATCH_JUNIOR_USER=junior
export SERVERWATCH_DISCORD_WEBHOOK_FILE="$HOME/.config/serverwatch-alerts/discord-webhook"
SERVERWATCH_PROJECT_DIR="$HOME/serverwatch" bash ~/setup.sh ELASTIC_IP
```

| Command | What it does |
|---|---|
| `git clone … ~/serverwatch` | Puts the code in the `ubuntu` user's home, owned by that user |
| `scp` | Copies a file to the server over the same encrypted SSH connection |
| `SERVERWATCH_PROJECT_DIR=…` | Tells the copied script where the cloned project is |
| `bash setup.sh ELASTIC_IP` | Installs packages, PostgreSQL, migrations, the API and collector services, React and Nginx, Cron backups, Docker, the accounts, the Discord timer and the `sw-*` aliases |

**Say (while setup runs):**
- "I clone into the home folder of the user that runs the services. That user owns the
  files, so no service needs root, and `git pull` works without sudo."
- "The script is safe to rerun: it keeps existing passwords and data. That is also how I
  update the server."
- "It generates every password and token itself, once, as files only that user can read
  (mode 600). Nothing secret is in Git."
- "It creates two accounts with different rights: senior can write to prod, junior only to
  dev. That is least privilege, enforced with groups and ACLs."
- "The aliases `sw-status`, `sw-logs` and `sw-timers` save typing; `sw-help` lists them."

In [ ]:
checkpoint("4")

## 5. Verify, Nginx by hand, public Swagger — G7, G8 (6 min)

```bash
bash verify.sh ELASTIC_IP          # read-only checks; expect 8 passed, 0 failed
```

Now the manual part. Open the Nginx site the script installed:

```bash
cd /etc/nginx/sites-enabled
ls -l
sudo nano serverwatch.conf
sudo nginx -t                      # check the syntax before applying
sudo systemctl reload nginx        # apply without dropping connections
```

**Explain the file:** `listen 80` is the public port; `root` serves the built React files;
the `location` for `/api`, `/docs` and `/openapi.json` forwards to
`proxy_pass http://127.0.0.1:8000`, the API on loopback.

In AWS, add **HTTP 80** to the security group. In the browser, open:
`http://ELASTIC_IP/`, `http://ELASTIC_IP/docs`, `http://ELASTIC_IP/api/v1/health?server_id=1`.

**Say:**
- "Nginx is a reverse proxy: the only public entry point. The API listens only on
  127.0.0.1, so it can't be reached from the internet directly."
- "`nginx -t` first, because a broken config would take the site down on reload."
- "Over plain HTTP the dashboard runs in demo mode. Live data needs a token, and tokens are
  only accepted over HTTPS, which I add in VG2."

In [ ]:
checkpoint("5")

## 6. systemd and the database — G7, G9 (4 min)

```bash
sw-status                                   # API and collector: active (running)
systemctl cat serverwatch-api               # the unit file
sudo ss -ltnp | grep -E ':5432|:8000'       # both bound to 127.0.0.1 only
```

**Say:**
- "systemd starts the API at boot and restarts it if it crashes. Same for the collector."
- "The API connects to PostgreSQL on loopback with its own limited database user. Tables
  are created by Alembic migrations, never by the app at start."
- "The health endpoint checks the database for real, which is why it returned `up` in the
  browser."

In [ ]:
checkpoint("6")

## 7. Cron backup and restore — G5 (6 min)

```bash
sudo cat /etc/cron.d/serverwatch-backup     # daily at 03:15
sudoedit /etc/cron.d/serverwatch-backup     # set it to the next minute for the demo
sudo journalctl -t serverwatch-backup -n 10 --no-pager
sudo ls -lt /var/backups/serverwatch
```

Then run the restore block from the [EC2 guide](ec2-manual-setup.md#verify-cron-backups-and-restore-g5)
into a separate database and show the row counts. Put the 03:15 schedule back.

| Cron field | `15 3 * * *` means |
|---|---|
| minute, hour | 03:15 |
| day, month, weekday | every day |

**Say:**
- "Cron runs a daily `pg_dump` and saves the last 24 hours of service logs."
- "A backup only counts if you can restore it, so I restore into a separate database and
  count the rows."
- "These backups live on the same disk. In production I would copy them off the server, for
  example with SCP from my own computer or to S3."

In [ ]:
checkpoint("7")

## 8. Discord alert — G6 (4 min)

```bash
sw-timers                                   # runs about once a minute
df -h /                                     # note the current usage, e.g. 31%
bash infrastructure/bootstrap/setup_discord_alerts.sh \
  "$HOME/.config/serverwatch-alerts/discord-webhook" 25
```

Wait for the message in Discord, then put the normal threshold back with the same command
and `90`.

**Say:**
- "A systemd timer checks disk usage every minute and posts to Discord when it crosses the
  threshold. I lower the threshold to trigger it safely."
- "It sends one alert per breach, not one per minute, and rearms when usage drops again."
- "I used a systemd timer instead of Cron here because systemd can hand the webhook to only
  this service, as a protected credential."

In [ ]:
checkpoint("8")

## 9. Docker Compose and persistence — VG1 (7 min)

```bash
cd ~/serverwatch
bash scripts/deployment/compose_host.sh start          # build images, start db → migrate → api → web
bash scripts/deployment/compose_host.sh ps -a          # db/api/web healthy, migrate Exited (0)
bash scripts/deployment/compose_host.sh collect        # Ctrl+C after ~20 s
bash scripts/deployment/compose_host.sh exec db psql -U postgres -d serverwatch -c "SELECT count(*) FROM metrics;"
bash scripts/deployment/compose_host.sh down           # containers and networks removed, volume kept
bash scripts/deployment/compose_host.sh up -d --wait
bash scripts/deployment/compose_host.sh exec db psql -U postgres -d serverwatch -c "SELECT count(*) FROM metrics;"
```

The two counts must match.

| Term | One line |
|---|---|
| Dockerfile | The recipe for an image |
| Image | The built, read-only package |
| Container | A running copy of an image; disposable |
| Volume | Storage outside the container; `postgres_data` keeps the database |
| Compose | One file that starts all services in order, with networks and secrets |

| `psql` flag | Meaning |
|---|---|
| `-U postgres` | Log in as the database admin user |
| `-d serverwatch` | Use the `serverwatch` database |
| `-c "…"` | Run this one SQL command and exit |

**Say:**
- "Four services: PostgreSQL, a one-time migration job, the API and an Nginx web container.
  `migrate` exiting with 0 is success, not a crash."
- "Compose gives each service a name on a private network: the API reaches the database at
  `db`. The database network is internal and publishes no port."
- "Containers are disposable; the volume isn't. The count is the same after recreating
  everything, so the data lives in the volume."
- "The native deployment is still running beside it (port 8000 vs Compose's 8080). Same app,
  packaged so it runs the same way on any machine with Docker."

In [ ]:
checkpoint("9")

## 10. Domain and HTTPS — VG2 (5 min)

1. In Namecheap → **Advanced DNS**, change both `A` records (`@` and `www`) to the new
   Elastic IP. In AWS, add **HTTPS 443** to the security group.
2. Check DNS and show the "before" state:

```bash
getent hosts serverwatch.dev www.serverwatch.dev   # both must show ELASTIC_IP
```

   Open `https://serverwatch.dev`: it fails, because `.dev` is HTTPS-only in every browser
   and there is no certificate yet.

3. Get the certificate and switch the domain to HTTPS:

```bash
bash infrastructure/bootstrap/setup_https.sh serverwatch.dev ELASTIC_IP
sudo certbot certificates                           # domains and expiry date
systemctl list-timers certbot.timer                 # automatic renewal
sudo nano /etc/nginx/sites-enabled/serverwatch-domain.conf
```

4. In the browser: `https://serverwatch.dev` (padlock), `/docs`, then **Connect your API**
   with the reader and admin tokens and edit an incident.

| Script step | What happens |
|---|---|
| DNS check | Stops early if the domain doesn't point at this server yet |
| Install Certbot | `apt-get install certbot` |
| Ownership check | Nginx serves `/.well-known/acme-challenge/`; Let's Encrypt fetches a token from it to prove I control the domain |
| `certbot certonly --webroot` | Issues the certificate into `/etc/letsencrypt/live/serverwatch.dev/` |
| HTTPS site | Adds the `listen 443 ssl` block and redirects HTTP to HTTPS (`nginx -t` first) |
| Renewal check | `certbot renew --dry-run` and the `certbot.timer` |
| Verification | Health and Swagger over HTTPS with real certificate checks, plus the redirect |

**Say:**
- "DNS is the phone book: the A record tells the world that `serverwatch.dev` is my
  Elastic IP."
- "Let's Encrypt proves I control the domain by fetching a file over port 80, then gives me
  a free certificate. Certbot renews it automatically through a systemd timer."
- "Certbot only issues the certificate. My own Nginx file uses it, so rerunning setup never
  removes HTTPS."
- "`.dev` is HTTPS-only in every browser, so without this certificate the site wouldn't load
  at all."
- "Now tokens travel encrypted, so the dashboard accepts a real token and I can edit an
  incident live."

In [ ]:
checkpoint("10")

## 10b. Optional: deploy after CI — VG4 bonus (4 min)

Only if time allows and the setup in [continuous deployment](continuous-deployment.md) is
done (secrets set, packages public, port 22 open to GitHub). Make a tiny visible change,
push it, and show **Actions**: CI, then *ServerWatch deploy*.

```bash
cat ~/.config/serverwatch-delivery/deployed-commit   # the SHA that GitHub deployed
bash scripts/deployment/compose_host.sh ps           # images tagged with that SHA
```

**Say:**
- "Deploy only runs after CI passes on main, so untested code never reaches the server."
- "Images are tagged with the commit SHA, so I know exactly which code runs."
- "GitHub's SSH key is restricted: it can't open a shell, it can only run my deploy script
  with a commit ID."

In [ ]:
checkpoint("10b")

## 11. Wrap-up (2 min)

**Say:** what you built, one thing you would do next (off-server backups, AWS Secrets
Manager, monitoring more than disk), then stop the recording.

Afterwards: watch the video, check the link, **stop or terminate the instance** and release
the Elastic IP. Narrow SSH again if it was opened for GitHub.

In [ ]:
checkpoint("11")

After a rehearsal, see how long each step took:

In [ ]:
summary()

## Likely questions

| Question | Short answer |
|---|---|
| Why not run as root? | A bug or break-in would control the whole server. Services run as a normal user. |
| Why loopback for the API and database? | Only Nginx is public; everything else is unreachable from outside. |
| Why three tokens? | Least privilege: the collector can't read or delete, a reader can't write. |
| Why migrations instead of creating tables at start? | Schema changes are versioned, reviewable and can be rolled back. |
| Native or Docker — which is better? | Docker is more portable and reproducible; native shows what Docker automates. |
| What if the server dies? | Backups are on the same disk today; next step is copying them off the host. |
| How do you know Compose, not native, answered? | Compose web is on 127.0.0.1:8080, native API on :8000; `ss -ltnp` and `compose_host.sh ps` show which. |
| Why Certbot `--webroot` instead of `--nginx`? | Certbot never edits my Nginx files, so setup can be rerun safely. |